In [1]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_34_try_2.pickle

In [3]:
%%RecordEvent
%%time
### cell 35 ###

def grab_subset_of_data_47(df, question):
    return (df.filter(like=question, axis=1)
            .rename(columns=lambda c: c.split('-')[-1].lstrip())
            .dropna(how='all'))


def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_47(
        question_of_interest, include_2017=False):
    mapping = [
        ('2022', responses_df_2022_cell10),
        ('2021', responses_df_2021),
        ('2020', responses_df_2020),
        ('2019', responses_df_2019_cell10),
        ('2018', responses_df_2018_cell10),
    ]
    if include_2017:
        mapping.append(('2017', responses_df_2017))
    # sort years lexicographically for chronological order
    mapping = sorted(mapping, key=lambda x: x[0])
    # build combined DataFrame with a single pass
    df_combined = pd.concat(
        (grab_subset_of_data_47(df_src, question_of_interest)
             .assign(year=year) for year, df_src in mapping),
        ignore_index=True
    )
    df_counts = df_combined.groupby('year', sort=True).count().reset_index()
    return df_combined, df_counts


def convert_df_of_counts_to_percentages_47(df, df_counts):
    years = sorted(df_counts['year'].astype(str).tolist())
    total_by_year = df['year'].value_counts().reindex(years)
    pct = (
        df_counts.set_index('year')
                 .reindex(years)
                 .div(total_by_year, axis=0)
                 * 100
    ).reset_index()
    return pct

# Update 2018 columns
question_of_interest_old = 'What machine learning frameworks have you used in the past 5 years?'
question_of_interest_new = 'Which of the following machine learning frameworks do you use on a regular basis?'
responses_df_2018_cell10.columns = (
    responses_df_2018_cell10.columns
        .str.replace(question_of_interest_old, question_of_interest_new, regex=False)
)
question_of_interest_cell47 = question_of_interest_new

# Combine multi-year subsets
ml_df_combined_cell47, ml_df_combined_counts_cell47 = (
    combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_47(
        question_of_interest_cell47
    )
)
ml_df_combined_2 = ml_df_combined_cell47.copy()

# Merge detailed framework columns into broader categories
groupings = {
    'TensorFlow/Keras': (
        ['TensorFlow', 'TensorFlow ', 'Keras', 'Keras '],
        'TensorFlow/Keras'
    ),
    'PyTorch/Lightning/Fast.ai': (
        ['PyTorch', 'PyTorch ', 'PyTorch Lightning ', 'Fast.ai ', 'Fastai'],
        'PyTorch/PyTorch Lightning/Fast.ai'
    ),
    'Xgboost/LightGBM/CatBoost': (
        ['Xgboost', 'Xgboost ', 'lightgbm', 'LightGBM ', 'catboost', 'CatBoost '],
        'Xgboost/LightGBM/CatBoost'
    ),
    'Scikit-learn': (
        ['Scikit—learn ', 'learn ', 'Learn'],
        'Scikit-learn'
    ),
}
for new_col, (cols, label) in groupings.items():
    mask = ml_df_combined_2[cols].notna().any(axis=1)
    ml_df_combined_2[new_col] = mask.map({True: label})
# drop intermediate columns
ml_df_combined_2.drop(
    columns=[col for cols, _ in groupings.values() for col in cols],
    inplace=True
)

# Recompute counts and percentages
ml_df_combined_counts_2 = ml_df_combined_2.groupby('year', sort=True).count().reset_index()
ml_df_combined_percentages = convert_df_of_counts_to_percentages_47(
    ml_df_combined_2, ml_df_combined_counts_2
)
# select and order columns
final_cols = [
    'year', 'Scikit-learn', 'TensorFlow/Keras',
    'PyTorch/Lightning/Fast.ai', 'Xgboost/LightGBM/CatBoost'
]
ml_df_combined_percentages = ml_df_combined_percentages[final_cols]

# melt for analysis or plotting
df_cell47 = ml_df_combined_percentages.melt(
    id_vars=['year'], var_name='', value_name='value'
).sort_values(['year', 'value'], ascending=True)

df_cell47.info()

<class 'pandas.core.frame.DataFrame'>
Index: 20 entries, 10 to 4
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   year    20 non-null     object 
 1           20 non-null     object 
 2   value   20 non-null     float64
dtypes: float64(1), object(2)
memory usage: 640.0+ bytes
CPU times: user 71.4 ms, sys: 6.6 ms, total: 78 ms
Wall time: 78 ms


In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_35_try_0.pickle